# Online Retail - Data Understanding & Assessment

# Objective

The objective of this notebook is to understand the dataset structure, identify data quality issues, and define an appropriate cleaning strategy before performing any analysis.

In [ ]:
import numpy as np
import pandas as pd

# Loading the dataset

In [ ]:
df = pd.read_csv('/content/OnlineRetail.csv', encoding='latin1')

# Column Description

- **InvoiceNo** - a 6-digit integral number uniquely assigned to each transaction.
- **StockCode** - a 5-digit alphanumeric identifier assigned to every distinct product.
- **Description** - textual, human-readable name of the purchased item.
- **Quantity** - the numerical count of a specific product item purchased or returned in a single transaction row.
- **InvoiceDate** - marks the exact date and time a transaction was recorded.
- **UnitPrice** - represents the price of a single unit of a product.
- **CustomerID** - a unique, 5-digit integral identifier assigned to each distinct buyer.
- **Country** - denotes the name of the country where a buyer resides.

# Answering some questions

###Number of rows and columns

In [ ]:
df.shape

(541909, 8)

###Data types of each column & Missing values in each column
**Conclusions**
- 1454 Null values found in col - `Description`
- 135080 Null values found in col - `CustomerID`

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


###Summary statistics

In [ ]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


###Number of duplicate rows
**Conclusion**
- The dataset contains 5,268 duplicate rows, representing approximately 0.97% of the dataset.
- Inspection of duplicate records showed that all attributes are identical across duplicate pairs, including transaction details, product information, quantity, pricing, customer identifier, and country.
- The duplicate records appear to be redundant copies rather than distinct business transactions. Since all column values are identical, these duplicates can be safely removed during the data cleaning phase without losing information.

In [ ]:
df.duplicated().sum()

np.int64(5268)

# Data Quality Assessment

## InvoiceNo Assessment
**Conclusions**
- Some InvoiceNo values start with the character "C" instead of being a 6-digit numeric value. All such records contain negative quantities. This is not a data quality issue. The "C" prefix appears to denote cancelled transactions or credit notes (product returns). These records represent legitimate business events and should be handled according to the analysis objective rather than automatically removed.

## Description Assessment
**Conclusions**
- Of the 1454 missing Description values, 1033 can be confidently imputed using StockCode. Additional validation showed that these recoverable StockCodes are not associated with suspicious descriptions such as "check", "amazon", or "manual", increasing confidence in the imputation strategy.
- An analysis of the Description column identified 127 unique lowercase descriptions that differ from the standard product naming convention. Examples include "check", "damages", "faulty", "broken", "returned", "missing", and "wrong barcode".
- These descriptions do not represent actual products. Instead, they appear to be related to inventory management, stock adjustments, product returns, damaged goods, administrative activities, and other operational processes.
- As a result, these records should be treated separately from normal product transactions during data cleaning and business analysis.

In [ ]:
sc_with_nan_desc = df[df['Description'].isnull()]['StockCode']

In [ ]:
temp = df.groupby('StockCode')['Description'].nunique()
new_df = temp[temp==1]
sc_with_one_desc = df[df['StockCode'].isin(new_df.index)]['StockCode']

In [ ]:
sc_with_replaceable_desc = sc_with_nan_desc[sc_with_nan_desc.isin(sc_with_one_desc)].values

## Quantity Assessment
**Conclusions**
- No missing values.
- No zero quantities.
- 10,624 negative quantities found.
- All negative quantities are associated with InvoiceNo beginning with "C".
- Extreme quantity values correspond to valid purchase-cancellation pairs.
- Therefore, negative quantities appear to represent returns/cancellations and are treated as business events rather than data quality issues.

In [ ]:
df[(df['Quantity']<0)].shape[0]

10624

In [ ]:
df[(df['Quantity']<0)]['InvoiceNo'].str.startswith('C').shape[0]

10624

## InvoiceDate Assessment
**Conclusions**
- Stored as object instead of datetime
- Successfully convertible to datetime
- No invalid dates found
- One-year transaction period

In [ ]:
df['InvoiceDate'].dtype

dtype('O')

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

## UnitPrice Assessment
**Conclusions**
- 2515 records have UnitPrice equal to zero. Investigation revealed two distinct patterns: (1) operational/system-generated records with missing descriptions and customer identifiers, and (2) legitimate customer transactions involving identifiable products and customers. Therefore, zero-price records should not be automatically treated as erroneous data.

In [ ]:
df[df['UnitPrice']<0].shape[0]

2

In [ ]:
df[df['UnitPrice']==0].shape[0]

2515

In [ ]:
df[(df['UnitPrice']==0) & ~(df['Description'].isnull())].shape[0]

1061

In [ ]:
df[(df['UnitPrice']==0) & ~(df['CustomerID'].isnull())]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,12/5/2010 14:02,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,12/16/2010 14:36,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,12/21/2010 13:45,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,1/6/2011 16:41,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,1/6/2011 16:41,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,1/13/2011 15:10,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2/10/2011 13:08,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,3/23/2011 10:25,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,3/30/2011 12:45,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,4/4/2011 14:42,0.0,14410.0,United Kingdom


In [ ]:
df['UnitPrice'].describe()

,UnitPrice
count,541909.000000
mean,4.611114
std,96.759853
min,-11062.060000
25%,1.250000
50%,2.080000
75%,4.130000
max,38970.000000


## CustomerID Assessment
**Conclusions**
- The CustomerID column contains 135,080 missing values, making it one of the most significant data quality concerns in the dataset. Investigation revealed that some of these missing values are associated with operational and inventory-related records containing descriptions such as "check", "amazon", "damages", "lost", and "missing". These records generally have no identifiable customer and often have a UnitPrice of zero.

- However, further analysis showed that the majority of records with missing CustomerID values are normal retail transactions containing valid product descriptions, positive prices, and realistic quantities. This suggests that the absence of CustomerID is not solely due to data errors. It is likely that many of these transactions were made by unregistered customers, guest buyers, or customers whose information was not recorded.

- Therefore, while the missing CustomerID values represent a data quality issue, they also reflect a business characteristic of the dataset rather than simple data corruption.

In [ ]:
round(df['CustomerID'].isnull().sum()/len(df['CustomerID'])*100, 2)

np.float64(24.93)

In [ ]:
df[(df['CustomerID'].isnull()) & (df['Description'].str.islower())].shape[0]

493

In [ ]:
df[(df['CustomerID'].isnull()) & ( (df['Description'].str.islower()) | (df['Description']=='?') )].shape[0]

540

In [ ]:
mask = ~(df['Description'].isnull())

In [ ]:
df[(df['CustomerID'].isnull()) & (df['UnitPrice']>0) & ( (mask) | (df[mask]['Description'].str.islower()) )]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,12/1/2010 14:32,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,12/1/2010 14:32,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,12/1/2010 14:32,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,12/9/2011 10:26,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,12/9/2011 10:26,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,12/9/2011 10:26,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,12/9/2011 10:26,10.79,NaN,United Kingdom


## Country Assessment
**Conclusions**
- The Country column contains 38 unique values and does not contain any missing values. Most country names are consistently formatted and represent valid geographic locations, indicating that the column is generally clean and reliable for analysis.
- Further investigation identified two special values: **"Unspecified"** and **"European Community"**. The value **"Unspecified"** appears in 446 records and is used when the customer's country information is unavailable. These records contain valid transactions and should therefore be treated as an unknown location category rather than invalid data.
- The value **"European Community"** appears in 61 records and is not an actual country. All 61 records belong to the same customer (CustomerID 15108), suggesting that it represents a business or geographic classification rather than a specific country.
- Overall, the Country column is of high quality with no major data quality issues. The only minor concern is the presence of non-country labels such as **"Unspecified"** and **"European Community"**, which should be handled separately when performing country-wise or geographic analysis.


In [ ]:
df[df['Country']=='Unspecified'].shape[0]

446

In [ ]:
df[df['Country']=='Unspecified']['CustomerID'].isnull().sum()

np.int64(202)

In [ ]:
df[df['Country']=='European Community'].shape[0]

61

In [ ]:
df[(df['Country']=='European Community') & (df['CustomerID']==15108.0)].shape[0]

61

# Assessment Summary

## Key Issues Identified

- 5,268 duplicate records.
- 1,454 missing Description values.
- 135,080 missing CustomerID values.
- InvoiceDate stored as object datatype.
- StockCode–Description inconsistency identified.
- Operational records mixed with product transactions.

## Cleaning Plan

- Remove duplicate records.
- Convert InvoiceDate to datetime.
- Impute recoverable Description values.
- Create a RecordType flag.
- Retain valid returns and cancellations.
- Preserve CustomerID missing values.